# StockIA — PoC: identificación de productos por foto (barcode + OCR fallback)

**Hipótesis a validar:**

> Una fotografía de productos etiquetados con códigos de barras grandes y de fondo blanco puede convertirse automáticamente en un listado estructurado de productos mediante lectura de barcode y OCR de respaldo, sin necesidad de reconocimiento visual del packaging.

Este notebook es un **PoC técnico** (prueba de concepto), no un sistema de producción. Se puede ejecutar de punta a punta en Google Colab.

## Pipeline

```text
                     FOTO
                       │
                       ▼
              PREPROCESAMIENTO
                       │
                       ▼
             DETECCIÓN BARCODE
                       │
              ┌────────┴────────┐
              │                 │
          DECODIFICADO        FALLA
              │                 │
              │                 ▼
              │                OCR
              │                 │
              │                 ▼
              └────────────► CÓDIGO
                               │
                               ▼
                    VALIDACIÓN / NORMALIZACIÓN
                               │
                               ▼
                         LOOKUP CATÁLOGO
                               │
                    ┌──────────┴──────────┐
                    │                     │
                 ENCONTRADO            NO ENCONTRADO
                    │                     │
                    ▼                     ▼
                 PRODUCTO              MARCAR ERROR
                    │
                    ▼
              AGRUPAR CANTIDADES
                    │
                    ▼
           RESULTADO TIPO STOCKIA
```

## Alcance: qué se usa y qué no

**Método primario:** decodificación de barcode (localización + lectura del código de barras).

**Fallback:** OCR sobre la línea de dígitos legibles debajo del barcode (o sobre la imagen completa cuando ni siquiera se localiza un barcode).

**Fuente de verdad:** un catálogo de productos (Open Food Facts si está disponible, catálogo local de respaldo si no).

**Deliberadamente NO se usa** un detector tipo YOLO ni un LLM para la identificación primaria de producto: la hipótesis específica que este PoC evalúa es si el par *barcode + OCR fallback* alcanza como mecanismo principal, así que evitamos mezclar esa señal con reconocimiento visual del packaging.

## Investigación previa (resumen)

Antes de escribir código se verificaron, contra documentación oficial, las piezas de este pipeline:

- **Open Food Facts**: consulta pública por GTIN mediante API v3.6, sin API key, con User-Agent identificable. Límite documentado de 15 lecturas/minuto/IP. Ver [documentación oficial](https://openfoodfacts.github.io/openfoodfacts-server/api/).
- **zxing-cpp** (`pip install zxing-cpp`, paquete `zxingcpp`): confirmado contra PyPI — bindings de Python del motor ZXing-C++, multi-formato (EAN-13, EAN-8, UPC-A, UPC-E, Code128, etc.), sin dependencia de una librería de sistema (a diferencia de `pyzbar`, que necesita `libzbar0`). Se probó localmente: decodifica correctamente, expone `text`, `format`, `position` (4 puntos) y `valid` (si el checksum interno es válido), pero **no expone un score de confianza** — por eso este notebook nunca inventa uno para el método barcode.
- **PaddleOCR 3.x** (`pip install paddleocr` + `paddlepaddle`): confirmado contra la documentación oficial (`paddleocr.github.io` / repo `PaddlePaddle/PaddleOCR`, ramas `main` y `version3.x`). La API actual es `PaddleOCR(...).predict(imagen)`, que devuelve objetos tipo diccionario con las claves `rec_texts` (textos reconocidos) y `rec_scores` (confianza por línea, sí provista por la librería). Instalar `paddlepaddle` es la parte más pesada del setup (puede tardar varios minutos en Colab).
- **OpenCV** y **Google Colab** (`google.colab.files`, `google.colab.userdata`): confirmados, son estables y ampliamente documentados.

Estas verificaciones condicionaron decisiones de diseño concretas más abajo (por ejemplo, por qué se eligió zxing-cpp sobre pyzbar, y por qué el cliente de Open Food Facts es un *stub* configurable en vez de una integración cerrada).


## 1. Instalación

La instalación se separa en dos celdas porque `paddlepaddle` es bastante más pesada que el resto (puede tardar varios minutos la primera vez). Si algo falla acá, el notebook lo va a reportar más adelante en vez de romperse silenciosamente — ver la Sección 10 (OCR fallback).

In [ ]:
# --- Dependencias livianas ---
# zxing-cpp: decodificación de barcode (motor ZXing-C++, multi-formato, sin
#            dependencia de una librería de sistema).
# opencv-python-headless: preprocesamiento de imágenes.
# python-barcode: generación de imágenes de barcode sintéticas para las
#                 pruebas de la Sección 7 (no se usa para decodificar nada).
# pillow, numpy, pandas, matplotlib, requests: utilitarios generales.
!pip -q install zxing-cpp opencv-python-headless pillow numpy pandas matplotlib requests python-barcode
print("Dependencias livianas instaladas.")

In [ ]:
# --- PaddleOCR (fallback OCR) ---
# Instalación más pesada. Se separa en dos pasos siguiendo la guía oficial:
# primero el motor (paddlepaddle, versión CPU), después el paquete paddleocr.
!pip -q install paddlepaddle
!pip -q install paddleocr
print("PaddleOCR instalado (si no hubo errores arriba).")

**Opcional** — solo para la comparación breve de la Sección 8. `pyzbar` necesita la librería de sistema `libzbar0`, que es justamente lo que zxing-cpp evita. Si esta celda falla, el resto del notebook sigue funcionando igual: la comparación simplemente va a mostrar `pyzbar` como no disponible.

In [ ]:
# Opcional: pyzbar, solo para comparar contra zxing-cpp en la Sección 8.
!apt-get -qq install -y libzbar0 > /dev/null 2>&1
!pip -q install pyzbar
print("pyzbar instalado (opcional).")

## 2. Imports

Cada import pesado (`zxingcpp`, `pyzbar`, `PaddleOCR`) queda protegido con `try/except` y una bandera booleana. La idea es que si algo no se instaló bien en la Sección 1, el notebook lo diga explícitamente en vez de fallar de golpe muchas celdas después.

In [ ]:
import io
import os
import re
import json
import time
import random
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

import cv2
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter

import barcode as barcode_gen  # python-barcode, solo para generar imágenes de prueba
from barcode.writer import ImageWriter

try:
  import zxingcpp
  ZXINGCPP_AVAILABLE = True
except ImportError as e:
  ZXINGCPP_AVAILABLE = False
  print(f"[AVISO] zxing-cpp no disponible: {e}")

try:
  from pyzbar.pyzbar import decode as pyzbar_decode
  PYZBAR_AVAILABLE = True
except Exception as e:
  PYZBAR_AVAILABLE = False
  print(f"[AVISO] pyzbar no disponible (opcional, no afecta el pipeline principal): {e}")

try:
  from paddleocr import PaddleOCR
  PADDLEOCR_IMPORT_OK = True
except Exception as e:
  PADDLEOCR_IMPORT_OK = False
  print(f"[AVISO] PaddleOCR no disponible: {e}")

random.seed(7)
np.random.seed(7)

print("ZXINGCPP_AVAILABLE  =", ZXINGCPP_AVAILABLE)
print("PYZBAR_AVAILABLE    =", PYZBAR_AVAILABLE)
print("PADDLEOCR_IMPORT_OK =", PADDLEOCR_IMPORT_OK)

if not ZXINGCPP_AVAILABLE:
  raise RuntimeError(
      "zxing-cpp es el decodificador de barcode primario de este PoC y no está "
      "disponible. Revisá la Sección 1 (instalación) antes de seguir."
  )

## 3. Configuración

Open Food Facts permite lecturas públicas sin API key. La configuración local se lee de `backend/.env`. El backend y este notebook usan API v3.6; identifican StockIA mediante User-Agent y respetan 15 consultas por minuto por proceso. Otros procesos que compartan IP también consumen ese límite.


In [ ]:
try:
  from google.colab import files as colab_files
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

if not IN_COLAB:
  from pathlib import Path
  from dotenv import load_dotenv
  load_dotenv(Path.cwd() / "backend" / ".env")
  load_dotenv(Path.cwd().parent / "backend" / ".env")

OPENFOODFACTS_BASE_URL = os.getenv("OPENFOODFACTS_BASE_URL", "https://world.openfoodfacts.org").rstrip("/")
OPENFOODFACTS_USER_AGENT = os.getenv("OPENFOODFACTS_USER_AGENT", "StockIA/1.0 (local development)")
OPENFOODFACTS_TIMEOUT_S = min(8, max(0.02, float(os.getenv("OPENFOODFACTS_TIMEOUT_MS", "5000")) / 1000))
WORKDIR = "stockia_poc_data"
IMG_DIR = os.path.join(WORKDIR, "imagenes")
OUT_DIR = os.path.join(WORKDIR, "resultados")
for d in (WORKDIR, IMG_DIR, OUT_DIR):
  os.makedirs(d, exist_ok=True)
print("WORKDIR:", os.path.abspath(WORKDIR))


## 4. Utilidades

Helpers generales: carga/visualización de imágenes, un timer simple, y las constantes de **estado** que va a llevar cada detección a lo largo del pipeline. Ningún estado implica una confianza inventada — ver la Sección 8 para el detalle de por qué el método barcode no reporta confidence y la Sección 10 para por qué el método OCR sí.

In [ ]:
# --- Estados posibles de una detección (Sección 17 del pedido original) ---
STATUS_IDENTIFIED = "IDENTIFIED"            # barcode decodificado + producto encontrado en catálogo
STATUS_OCR_IDENTIFIED = "OCR_IDENTIFIED"    # candidato vía OCR + producto encontrado en catálogo
STATUS_NOT_FOUND = "NOT_FOUND"              # código válido, pero no está en ningún catálogo
STATUS_INVALID_BARCODE = "INVALID_BARCODE"  # candidato numérico que no pasa el checksum
STATUS_LOW_CONFIDENCE = "LOW_CONFIDENCE"    # lecturas contradictorias sobre la misma región
STATUS_NO_CODE_DETECTED = "NO_CODE_DETECTED"  # no se localizó ni un barcode ni un candidato OCR
STATUS_ERROR = "ERROR"                      # error inesperado durante el procesamiento


def load_image(path_or_array):
  '''Carga una imagen BGR desde un path, o deja pasar un array ya cargado.'''
  if isinstance(path_or_array, str):
    img = cv2.imread(path_or_array)
    if img is None:
      raise FileNotFoundError(f"No se pudo leer la imagen: {path_or_array}")
    return img
  return path_or_array


def show_image(img_bgr, title="", ax=None):
  img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
  if ax is None:
    plt.figure(figsize=(5, 5))
    ax = plt.gca()
  ax.imshow(img_rgb)
  ax.set_title(title, fontsize=9)
  ax.axis("off")


def show_image_grid(images_with_titles, cols=3, figsize_per_cell=(3.2, 3.2)):
  n = len(images_with_titles)
  rows = (n + cols - 1) // cols
  fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_cell[0] * cols, figsize_per_cell[1] * rows))
  axes = np.array(axes).reshape(-1)
  for ax, (img, title) in zip(axes, images_with_titles):
    show_image(img, title=title, ax=ax)
  for ax in axes[n:]:
    ax.axis("off")
  plt.tight_layout()
  plt.show()


class Timer:
  '''Context manager simple para medir tiempos de inferencia en milisegundos.'''

  def __enter__(self):
    self._t0 = time.perf_counter()
    self.elapsed_ms = None
    return self

  def __exit__(self, *exc_info):
    self.elapsed_ms = (time.perf_counter() - self._t0) * 1000.0


def normalize_barcode(text):
  '''Limpia espacios/guiones y devuelve la corrida de dígitos más larga.

  Deliberadamente NO concatena dígitos separados por un caracter no numérico
  en el medio (p. ej. un caracter mal leído por OCR): eso alteraría el valor
  del código en vez de solo limpiarlo. Ver Sección 11 para el detalle.
  '''
  if not text:
    return ""
  cleaned = re.sub(r"[\s\-]", "", text)
  digit_runs = re.findall(r"\d+", cleaned)
  if not digit_runs:
    return ""
  return max(digit_runs, key=len)

## 5. Catálogo Open Food Facts

Consulta pública por identificador: `GET /api/v3.6/product/{barcode}.json` con `fields=code,product_name,product_name_es,product_name_en,brands`. Solo se envía el código, nunca la foto. El catálogo contiene alimentos y puede tener productos faltantes o datos incompletos. No aporta precios ni stock a StockIA.

[Documentación oficial](https://openfoodfacts.github.io/openfoodfacts-server/api/). Datos bajo ODbL / Database Contents License; cada producto conserva un enlace de atribución. Para staging usar `https://world.openfoodfacts.net` (Basic `off:off`).


In [ ]:
from collections import deque
from urllib.parse import urlsplit
from time import monotonic

openfoodfacts_results = {}  # snapshot para exportación; no se reutiliza entre consultas
openfoodfacts_request_count = 0
openfoodfacts_reads = deque()
openfoodfacts_blocked_until = 0


def lookup_openfoodfacts(barcode):
  """Consulta pública sin cache; devuelve producto o None y conserva el catálogo local."""
  global openfoodfacts_request_count, openfoodfacts_blocked_until
  now = monotonic()
  while openfoodfacts_reads and openfoodfacts_reads[0] <= now - 60:
    openfoodfacts_reads.popleft()
  if now < openfoodfacts_blocked_until or len(openfoodfacts_reads) >= 15:
    print("[Open Food Facts] Límite de consultas; reintentá más tarde.")
    return None
  openfoodfacts_reads.append(now)
  headers = {"User-Agent": OPENFOODFACTS_USER_AGENT, "Accept": "application/json"}
  if urlsplit(OPENFOODFACTS_BASE_URL).hostname == "world.openfoodfacts.net":
    headers["Authorization"] = "Basic b2ZmOm9mZg=="
  url = OPENFOODFACTS_BASE_URL + f"/api/v3.6/product/{barcode}.json"
  try:
    openfoodfacts_request_count += 1
    resp = requests.get(url, headers=headers, params={
      "fields": "code,product_name,product_name_es,product_name_en,brands", "lc": "es",
    }, timeout=OPENFOODFACTS_TIMEOUT_S, allow_redirects=False)
    if resp.status_code == 429:
      try:
        wait = max(60, int(resp.headers.get("Retry-After", "60")))
      except ValueError:
        wait = 60
      openfoodfacts_blocked_until = monotonic() + wait
    if resp.status_code != 200:
      print(f"[Open Food Facts] {barcode}: HTTP {resp.status_code}")
      return None
    data = resp.json()
    food = data.get("product", {})
    code = str(food.get("code") or "")
    if data.get("status") != "success" or data.get("result", {}).get("id") != "product_found" or code.zfill(14) != barcode.zfill(14):
      return None
    product = {
      "barcode": barcode, "product_id": code, "sku": barcode,
      "name": food.get("product_name_es") or food.get("product_name") or food.get("product_name_en"),
      "brand": food.get("brands"), "source": "openfoodfacts",
      "source_url": f"https://world.openfoodfacts.org/product/{code}",
    }
    openfoodfacts_results[barcode] = product
    return product
  except (requests.RequestException, ValueError, TypeError):
    print(f"[Open Food Facts] No se pudo consultar {barcode}.")
    return None


## 6. Catálogo local de respaldo

Los identificadores pueden faltar en Open Food Facts. Este catálogo demo permite ejecutar el PoC incluso sin red. Sus nombres y marcas son fixtures del notebook; no deben confundirse con datos descargados ni importarse automáticamente al inventario.


In [ ]:
import re

# --- Catálogo Local Consolidado ---
LOCAL_CATALOG = {
    # Gaseosas
    "7798113300218": {"product_id": "cde341c8-8f49-40b1-95d2-e5aa2e9e36e4", "sku": "GASEOSA-MANAOS-LIMA", "name": "Gaseosa de Lima Limón", "brand": "Manaos", "source": "local"},
    "7798113300010": {"product_id": "gas-mn-cola", "sku": "GASEOSA-MANAOS-COLA", "name": "Gaseosa Cola", "brand": "Manaos", "source": "local"},
    "7791813828464": {"product_id": "gas-pr-cola", "sku": "GASEOSA-PRITY-COLA", "name": "Gaseosa Cola", "brand": "Prity", "source": "local"},
    "7792758372074": {"product_id": "86884b46-4313-4b82-b5bd-d306a2f7ae50", "sku": "GASEOSA-SECCO-LIMON", "name": "Gaseosa Limón Intenso", "brand": "Secco", "source": "local"},
    "7790639003567": {"product_id": "6f775029-aa42-429f-96af-4541ae9d53c5", "sku": "GASEOSA-CUNN-COLA", "name": "Gaseosa Cola", "brand": "Cunnington", "source": "local"},
    "7791913001859": {"product_id": "90e51297-576e-443b-a2d9-aab7901530e0", "sku": "GASEOSA-DOBLE-COLA", "name": "Gaseosa", "brand": "Doble Cola", "source": "local"},

    # Galletitas
    "7622202839962": {"product_id": "gall-bagley-surt", "sku": "GALL-SURT-VARIEDAD", "name": "Galletitas Surtidas", "brand": "Bagley", "source": "local"},
    "7790040047006": {"product_id": "gall-bagley-salv", "sku": "GALL-SALVADO-HOGARE", "name": "Galletitas de Salvado", "brand": "Bagley", "source": "local"},
    "7790040147720": {"product_id": "gall-surt-promo", "sku": "GALL-SURTIDAS-BAG", "name": "Galletitas Surtidas", "brand": "Bagley", "source": "local"},
    "7793890064797": {"product_id": "gall-granix-salv", "sku": "GALL-SALVADO-GRANIX", "name": "Galletitas de Salvado", "brand": "Granix", "source": "local"},
    "7790040258402": {"product_id": "gall-generic-1", "sku": "GALL-GENERICA", "name": "Galletitas", "brand": "Desconocida", "source": "local"},
    "7622201735296": {"product_id": "gall-oreo-reg", "sku": "GALL-OREO", "name": "Galletitas", "brand": "Oreo", "source": "local"}
}

def lookup_local(barcode):
  product = LOCAL_CATALOG.get(barcode)
  return dict(product) if product else None

def lookup_product(barcode):
  """Lógica mejorada: Open Food Facts (si tiene datos útiles) > Local"""
  product = lookup_openfoodfacts(barcode)
  # Si el producto existe en Open Food Facts Y tiene un nombre asignado, lo usamos
  if product and product.get('name'):
    return product, "openfoodfacts"

  # De lo contrario (o si Open Food Facts falló), buscamos en Local
  product = lookup_local(barcode)
  if product:
    return product, "local"

  return None, "not_found"

# Verificación rápida con el fix
_test_bc = "7798113300010"
_p, _s = lookup_product(_test_bc)
print(f"Prueba {_test_bc} {_s} -> {_p['name'] if _p else 'No encontrado'}")

## 7. Carga de imágenes

Tres modos, tal como se pidió:

- **A. Imagen subida por el usuario** (`google.colab.files.upload()`), para probar con fotos reales.
- **B. Imagen desde una URL.**
- **C. Imágenes sintéticas de prueba**, generadas con `python-barcode` + PIL, para poder validar el pipeline completo aunque todavía no haya fotos reales.

Las celdas A y B son opcionales — corré la que te sirva. La celda C se ejecuta siempre porque de ahí salen los casos de evaluación de la Sección 16.

In [ ]:
def upload_image_from_computer():
  '''Modo A: subir una imagen real desde tu computadora (solo funciona en Colab).'''
  if not IN_COLAB:
    print("Este modo requiere Google Colab (google.colab.files). Salteando.")
    return None
  from google.colab import files
  uploaded = files.upload()
  if not uploaded:
    print("No se subió ningún archivo.")
    return None
  filename = next(iter(uploaded.keys()))
  dest = os.path.join(IMG_DIR, filename)
  with open(dest, "wb") as f:
    f.write(uploaded[filename])
  print("Imagen guardada en:", dest)
  return dest


def download_image_from_url(url, filename=None):
  '''Modo B: descargar una imagen desde una URL.'''
  resp = requests.get(url, timeout=15)
  resp.raise_for_status()
  if filename is None:
    filename = "url_" + re.sub(r"[^a-zA-Z0-9_.]", "_", url.split("/")[-1] or "imagen.jpg")
  dest = os.path.join(IMG_DIR, filename)
  with open(dest, "wb") as f:
    f.write(resp.content)
  print("Imagen guardada en:", dest)
  return dest


# --- SUBIR FOTOS DE PRUEBA AQUÍ ---
# Descomentá la línea que corresponda para probar con una imagen real:
#
# ruta_real = upload_image_from_computer()
# ruta_real = download_image_from_url("https://ejemplo.com/foto.jpg")
#
# El resto del notebook funciona igual sin esto: usa las imágenes sintéticas
# generadas en la próxima celda.
print("Modos A/B disponibles como funciones: upload_image_from_computer(), download_image_from_url(url).")

### 7.1 Generación de imágenes sintéticas (modo C)

No inventamos productos reales: los tres códigos de abajo son los mismos ficticios del catálogo local (Sección 6). Cada caso está pensado para ejercitar una condición distinta del pipeline, siguiendo los 8 casos pedidos:

| Caso | Qué prueba | Cómo se genera |
|---|---|---|
| A | Barcode completamente visible | Barcode limpio, sin degradación |
| B | Barcode parcialmente deformado | Rotación 20° |
| C | Mala iluminación | Oscurecido (×0.22) + ruido gaussiano (σ=40) |
| D | Barcode pequeño | Reducido a 15% de su tamaño e insertado en un encuadre más grande |
| E | Número legible pero barcode difícil | Blur (σ=6) + ruido solo en la zona de barras; el texto queda intacto |
| F | Varios productos | Composición de 3 barcodes en una sola imagen (2 unidades de un producto + 1 de otro) |
| G | Producto sin barcode visible | Solo texto de etiqueta, sin ningún barcode |
| H | Código inválido | Bloque de ruido en vez de barras reales + texto con un checksum EAN-13 inválido |

Todos los parámetros de degradación quedan guardados en `TEST_CASE_PARAMS` para que la Sección 22 (robustez) los pueda reportar tal cual se usaron.

In [ ]:
TEST_CASE_PARAMS = {}  # case_id -> dict con los parámetros de degradación usados


def _load_font(size):
  try:
    return ImageFont.truetype("DejaVuSans.ttf", size)
  except Exception:
    return ImageFont.load_default()


def gen_barcode_image(code12):
  '''Genera un EAN-13 con python-barcode a partir de un prefijo de 12 dígitos.

  Devuelve (imagen_PIL, codigo_final_con_checksum).
  '''
  b = barcode_gen.get("ean13", code12, writer=ImageWriter())
  buf = io.BytesIO()
  b.write(buf)
  buf.seek(0)
  img = Image.open(buf).convert("RGB")
  return img, b.ean


def compose_label(barcode_img, product_name=None, pad=20):
  '''Arma una "foto de producto": el barcode sobre fondo blanco, con un
  renglón de texto opcional arriba (nombre del producto).'''
  w, h = barcode_img.size
  extra_top = 40 if product_name else 0
  canvas = Image.new("RGB", (w + pad * 2, h + pad * 2 + extra_top), "white")
  if product_name:
    draw = ImageDraw.Draw(canvas)
    draw.text((pad, pad), product_name, fill="black", font=_load_font(22))
  canvas.paste(barcode_img, (pad, pad + extra_top))
  return canvas


def pil_to_bgr(img_pil):
  return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


def bgr_to_pil(img_bgr):
  return Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))


def darken(img_bgr, factor):
  return np.clip(img_bgr.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def add_gaussian_noise(img_bgr, sigma, seed=7):
  rng = np.random.default_rng(seed)
  noise = rng.normal(0, sigma, img_bgr.shape).astype(np.float32)
  return np.clip(img_bgr.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def rotate_image(img_bgr, angle_deg):
  h, w = img_bgr.shape[:2]
  M = cv2.getRotationMatrix2D((w // 2, h // 2), angle_deg, 1.0)
  return cv2.warpAffine(img_bgr, M, (w, h), borderValue=(255, 255, 255))

In [ ]:
def build_case_A():
  '''Barcode completamente visible: Manaos Lima (Local).'''
  code_prefix = "779811330021" # EAN-13 completo: 7798113300218
  img, code = gen_barcode_image(code_prefix)
  labeled = compose_label(img, "Manaos Lima 2.25L")
  TEST_CASE_PARAMS["A"] = {"degradacion": "ninguna"}
  return pil_to_bgr(labeled), [{"barcode": code, "quantity": 1}]

def build_case_B():
  '''Barcode rotado: Manaos Cola (Local/API).'''
  code_prefix = "779811330001" # EAN-13 completo: 7798113300010
  img, code = gen_barcode_image(code_prefix)
  labeled = compose_label(img, "Manaos Cola 2.25L")
  angle = 20
  rotated = rotate_image(pil_to_bgr(labeled), angle)
  TEST_CASE_PARAMS["B"] = {"degradacion": "rotacion", "angulo_grados": angle}
  return rotated, [{"barcode": code, "quantity": 1}]

def build_case_C():
  '''Mala iluminación: Bagley Surtidas (Local).'''
  code_prefix = "762220283996" # EAN-13 completo: 7622202839962
  img, code = gen_barcode_image(code_prefix)
  labeled = compose_label(img, "Bagley Variedad")
  factor, sigma = 0.22, 40
  degraded = add_gaussian_noise(darken(pil_to_bgr(labeled), factor), sigma)
  TEST_CASE_PARAMS["C"] = {"degradacion": "oscurecido+ruido", "factor_brillo": factor, "sigma_ruido": sigma}
  return degraded, [{"barcode": code, "quantity": 1}]

def build_case_D():
  '''Barcode pequeño: Coca-Cola (API).'''
  code_prefix = "779007041136" # EAN-13 completo: 7790070411365
  img, code = gen_barcode_image(code_prefix)
  labeled = compose_label(img, "Coca-Cola Original")
  scale = 0.15
  small = labeled.resize((int(labeled.width * scale), int(labeled.height * scale)), Image.LANCZOS)
  canvas_size = 500
  canvas = Image.new("RGB", (canvas_size, canvas_size), "white")
  canvas.paste(small, (canvas_size // 2 - small.width // 2, canvas_size // 2 - small.height // 2))
  TEST_CASE_PARAMS["D"] = {"degradacion": "escalado", "factor_escala": scale}
  return pil_to_bgr(canvas), [{"barcode": code, "quantity": 1}]

def build_case_E():
  '''Blur en barras pero texto legible: Galletitas Oreo (Local).'''
  code_prefix = "762220173529" # EAN-13 completo: 7622201735296
  img, code = gen_barcode_image(code_prefix)
  w, h = img.size
  bars_h = int(h * 0.68)
  bars_region = img.crop((0, 0, w, bars_h))
  text_region = img.crop((0, bars_h, w, h))
  blurred = bars_region.filter(ImageFilter.GaussianBlur(radius=6))
  recombined = Image.new("RGB", (w, h), "white")
  recombined.paste(blurred, (0, 0))
  recombined.paste(text_region, (0, bars_h))
  labeled = compose_label(recombined, "Galletitas Oreo")
  TEST_CASE_PARAMS["E"] = {"degradacion": "blur en barras"}
  return pil_to_bgr(labeled), [{"barcode": code, "quantity": 1}]

def build_case_F():
  '''Varios productos: 2 Manaos Lima + 1 Oreo.'''
  img_a, code_a = gen_barcode_image("779811330021")
  img_b, code_b = gen_barcode_image("762220173529")
  imgs = [compose_label(img_a, "Manaos Lima"), compose_label(img_a, "Manaos Lima"), compose_label(img_b, "Oreo")]
  pad = 30
  canvas = Image.new("RGB", (sum(im.width for im in imgs) + pad*4, max(im.height for im in imgs) + pad*2), "white")
  x = pad
  for im in imgs: canvas.paste(im, (x, pad)); x += im.width + pad
  gt = [{"barcode": code_a, "quantity": 2}, {"barcode": code_b, "quantity": 1}]
  return pil_to_bgr(canvas), gt

In [ ]:
def build_all_test_cases():
  builders = {
      "A": build_case_A, "B": build_case_B, "C": build_case_C, "D": build_case_D,
      "E": build_case_E, "F": build_case_F
  }
  descriptions = {
      "A": "Barcode visible (Manaos Lima)",
      "B": "Barcode rotado (Manaos Cola)",
      "C": "Mala iluminación (Bagley Surtidas)",
      "D": "Barcode pequeño (Coca-Cola)",
      "E": "Blur en barras (Oreo)",
      "F": "Varios productos (Manaos + Oreo)",
  }
  cases = {}
  for case_id, builder in builders.items():
    img_bgr, ground_truth = builder()
    path = os.path.join(IMG_DIR, f"case_{case_id}.png")
    cv2.imwrite(path, img_bgr)
    cases[case_id] = {
        "image_path": path,
        "image": img_bgr,
        "description": descriptions[case_id],
        "ground_truth": ground_truth,
    }
  return cases

TEST_CASES = build_all_test_cases()
print(f"Se generaron {len(TEST_CASES)} casos de prueba.")

In [ ]:
# Visualización corregida de los casos generados
show_image_grid(
    [(c["image"], f"Caso {cid}: {c['description']}") for cid, c in TEST_CASES.items()],
    cols=3
)

## 8. Detección y decodificación de barcode

**Elegimos zxing-cpp como decodificador primario.** Comparación breve contra las alternativas consideradas:

| Librería | Dependencia de sistema | Formatos | Notas |
|---|---|---|---|
| **zxing-cpp** (elegida) | Ninguna (wheel autocontenido) | EAN-13/8, UPC-A/E, Code128, QR, y más | Motor ZXing-C++, activamente mantenido, expone `position` y `valid`, no expone confianza |
| pyzbar | `libzbar0` (sistema) | EAN/UPC, Code128, QR, etc. | Muy usado, pero atado a ZBar (desarrollo más lento) y a instalar una lib de sistema — problemático en algunos entornos Colab |
| OpenCV (`cv2.barcode`) | Requiere `opencv-contrib-python` | 1D básicos | Menos maduro para EAN-13 en nuestras pruebas |

Nos quedamos con zxing-cpp porque es el que menos fricción de instalación tiene en Colab y el que mejor resultado dio en las pruebas. Igual dejamos pyzbar como comparación opcional si se instaló en la Sección 1.

**Sobre la confianza:** ninguna de estas librerías devuelve un score de confianza para barcode 1D. Este notebook nunca inventa uno — el campo `confidence` de una detección por barcode queda siempre en `None`.

In [ ]:
def bbox_from_position(position):
  '''Convierte los 4 puntos que devuelve zxing-cpp en un bbox (x, y, w, h).'''
  xs = [position.top_left.x, position.top_right.x, position.bottom_left.x, position.bottom_right.x]
  ys = [position.top_left.y, position.top_right.y, position.bottom_left.y, position.bottom_right.y]
  x_min, x_max = min(xs), max(xs)
  y_min, y_max = min(ys), max(ys)
  return {"x": int(x_min), "y": int(y_min), "w": int(x_max - x_min), "h": int(y_max - y_min)}


def decode_barcodes_zxing(img_bgr):
  '''Detección + decodificación con zxing-cpp.

  Devuelve una lista de dicts:
    {"value", "format", "bbox", "confidence": None, "valid"}
  confidence siempre es None: la librería no la provee (ver más arriba).
  '''
  results = zxingcpp.read_barcodes(img_bgr)
  out = []
  for r in results:
    out.append({
        "value": r.text,
        "format": str(r.format),
        "bbox": bbox_from_position(r.position),
        "confidence": None,
        "valid": bool(r.valid),
    })
  return out


def decode_barcodes_pyzbar(img_bgr):
  '''Decodificación con pyzbar, solo para la comparación de la Sección 17.'''
  if not PYZBAR_AVAILABLE:
    return None  # no instalado — no es lo mismo que "no encontró nada"
  results = pyzbar_decode(img_bgr)
  out = []
  for r in results:
    x, y, w, h = r.rect.left, r.rect.top, r.rect.width, r.rect.height
    out.append({
        "value": r.data.decode("utf-8", errors="ignore"),
        "format": str(r.type),
        "bbox": {"x": x, "y": y, "w": w, "h": h},
        "confidence": None,
    })
  return out


def _bbox_iou(a, b):
  ax0, ay0, ax1, ay1 = a["x"], a["y"], a["x"] + a["w"], a["y"] + a["h"]
  bx0, by0, bx1, by1 = b["x"], b["y"], b["x"] + b["w"], b["y"] + b["h"]
  ix0, iy0 = max(ax0, bx0), max(ay0, by0)
  ix1, iy1 = min(ax1, bx1), min(ay1, by1)
  iw, ih = max(0, ix1 - ix0), max(0, iy1 - iy0)
  inter = iw * ih
  if inter == 0:
    return 0.0
  area_a = a["w"] * a["h"]
  area_b = b["w"] * b["h"]
  return inter / float(area_a + area_b - inter)


def cluster_overlapping_detections(detections, iou_threshold=0.3):
  '''Agrupa detecciones cuyo bbox se superpone (misma región física del barcode).

  Esto existe porque, en las pruebas, encontramos casos reales (ver Sección 20 /
  Limitaciones) donde zxing-cpp devuelve DOS lecturas distintas —ambas con
  checksum válido— sobre la misma zona de una imagen rotada. Un solo checksum
  válido no alcanza para confiar ciegamente en una lectura: si dos lecturas
  que se superponen espacialmente no coinciden en el valor, señalamos la
  región como ambigua (estado LOW_CONFIDENCE) en vez de elegir una al azar.
  '''
  clusters = []
  used = [False] * len(detections)
  for i, det in enumerate(detections):
    if used[i]:
      continue
    group = [det]
    used[i] = True
    for j in range(i + 1, len(detections)):
      if used[j]:
        continue
      if _bbox_iou(det["bbox"], detections[j]["bbox"]) >= iou_threshold:
        group.append(detections[j])
        used[j] = True
    clusters.append(group)
  return clusters

## 9. Preprocesamiento para OCR

Cuando el barcode SÍ se localizó pero no se pudo decodificar del todo (o cuando queremos confirmar una región ambigua), recortamos la franja inmediatamente debajo del bbox — ahí es donde va el número legible.

Cuando el barcode **no se localizó en absoluto** (cero detecciones), no tenemos ningún bbox del cual partir. **Limitación reconocida:** zxing-cpp no expone una etapa de "localización sin decodificación" — es decodificación todo-en-uno. En ese caso el fallback corre OCR sobre la imagen completa y confía en la extracción de candidatos + validación de checksum (Sección 11) para filtrar el ruido. Ver Sección 20 (Limitaciones) para el detalle y una mejora posible a futuro.

Probamos varias variantes de preprocesamiento en vez de asumir que una sola funciona mejor, tal como se pidió.

In [ ]:
def crop_below_barcode(img_bgr, bbox, extra_frac=0.7, pad_x_frac=0.05):
  '''Recorta la franja debajo del bbox del barcode (ahí va el número legible).'''
  x, y, w, h = bbox["x"], bbox["y"], bbox["w"], bbox["h"]
  H, W = img_bgr.shape[:2]
  pad_x = int(w * pad_x_frac)
  x0 = max(0, x - pad_x)
  x1 = min(W, x + w + pad_x)
  y0 = max(0, y + h)
  y1 = min(H, y + h + int(h * extra_frac))
  if y1 <= y0 or x1 <= x0:
    return img_bgr  # bbox degenerado — devolvemos la imagen completa como respaldo
  return img_bgr[y0:y1, x0:x1]


def preprocess_variants(img_bgr):
  '''Devuelve un dict {nombre_variante: imagen} con las transformaciones a probar.'''
  gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
  h, w = gray.shape[:2]
  scale = 3 if max(h, w) < 200 else 1
  upscaled = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC) if scale > 1 else gray

  _, otsu = cv2.threshold(upscaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  adaptive = cv2.adaptiveThreshold(
      upscaled, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10
  )
  sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
  sharpened = cv2.filter2D(upscaled, -1, sharpen_kernel)
  denoised = cv2.fastNlMeansDenoising(upscaled, h=15)

  variants = {
      "original": img_bgr,
      "grises": cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
      "upscaled": cv2.cvtColor(upscaled, cv2.COLOR_GRAY2BGR),
      "threshold_otsu": cv2.cvtColor(otsu, cv2.COLOR_GRAY2BGR),
      "threshold_adaptativo": cv2.cvtColor(adaptive, cv2.COLOR_GRAY2BGR),
      "sharpen": cv2.cvtColor(sharpened, cv2.COLOR_GRAY2BGR),
      "denoise": cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR),
  }
  return variants


# Demo rápida sobre el caso E (barcode difícil, texto legible)
_demo_crop = TEST_CASES["E"]["image"]  # sin bbox conocido todavía: usamos la imagen completa
_variants = preprocess_variants(_demo_crop)
show_image_grid([(img, name) for name, img in _variants.items()], cols=4)

## 10. OCR fallback (PaddleOCR)

Usamos PaddleOCR como estaba pedido, configurado para el caso específico (leer una línea corta de dígitos), no como OCR genérico de documentos: desactivamos la clasificación de orientación de documento y el "unwarping", que están pensadas para páginas escaneadas.

Si PaddleOCR no llegó a instalarse bien (Sección 1), esta sección lo va a decir explícitamente y el resto del notebook sigue funcionando en modo "solo barcode" — no se oculta la falla.

In [ ]:
_paddle_ocr_engine = None
PADDLEOCR_READY = False

if PADDLEOCR_IMPORT_OK:
  try:
    _paddle_ocr_engine = PaddleOCR(
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        lang="es",
    )
    PADDLEOCR_READY = True
    print("PaddleOCR inicializado correctamente.")
  except Exception as e:
    print(f"[AVISO] PaddleOCR no se pudo inicializar: {e}")
else:
  print("[AVISO] PaddleOCR no está instalado — el pipeline va a funcionar solo con barcode.")


def run_ocr(img_bgr):
  '''Corre PaddleOCR sobre una imagen (o recorte) y devuelve (textos, scores).

  Los scores SÍ vienen de la librería (PaddleOCR los expone vía rec_scores),
  así que acá no estamos inventando una confianza — a diferencia del método
  de barcode (Sección 8), que no la provee.
  '''
  if not PADDLEOCR_READY:
    return [], []
  try:
    result = _paddle_ocr_engine.predict(img_bgr)
  except Exception as e:
    print(f"[AVISO] Error corriendo PaddleOCR: {e}")
    return [], []
  if not result:
    return [], []
  res = result[0]
  texts = list(res.get("rec_texts", []))
  scores = list(res.get("rec_scores", []))
  return texts, scores

## 11. Validación y extracción de candidatos

Del texto que devuelve OCR, extraemos candidatos numéricos y los validamos con el checksum real de cada formato (EAN-13, EAN-8, UPC-A). Una cadena de dígitos solo se acepta como código válido si el checksum cierra — no aceptamos cualquier secuencia numérica.

In [ ]:
def validate_ean13(code):
  if not (isinstance(code, str) and len(code) == 13 and code.isdigit()):
    return False
  digits = [int(c) for c in code]
  total = sum(d * (3 if i % 2 else 1) for i, d in enumerate(digits[:12]))
  check = (10 - (total % 10)) % 10
  return check == digits[12]


def validate_ean8(code):
  if not (isinstance(code, str) and len(code) == 8 and code.isdigit()):
    return False
  digits = [int(c) for c in code]
  total = sum(d * (3 if i % 2 == 0 else 1) for i, d in enumerate(digits[:7]))
  check = (10 - (total % 10)) % 10
  return check == digits[7]


def validate_upca(code):
  if not (isinstance(code, str) and len(code) == 12 and code.isdigit()):
    return False
  digits = [int(c) for c in code]
  total = sum(d * (3 if i % 2 == 0 else 1) for i, d in enumerate(digits[:11]))
  check = (10 - (total % 10)) % 10
  return check == digits[11]


VALIDATORS = {
    "EAN-13": validate_ean13,
    "EAN-8": validate_ean8,
    "UPC-A": validate_upca,
}


def classify_and_validate(code):
  '''A partir de un candidato ya normalizado (solo dígitos), intenta
  clasificarlo por longitud y valida su checksum.

  Devuelve (formato_o_None, es_valido).
  '''
  by_length = {13: "EAN-13", 8: "EAN-8", 12: "UPC-A"}
  fmt = by_length.get(len(code))
  if fmt is None:
    return None, False
  return fmt, VALIDATORS[fmt](code)


def extract_numeric_candidates(texts, scores=None):
  '''Recorre las líneas de texto reconocidas por OCR y arma la lista de
  candidatos: para cada línea, normaliza y se queda con la corrida de
  dígitos más larga (Sección 4). Devuelve una lista de dicts:
  {"raw_text", "candidate", "format", "valid", "confidence"}.

  "confidence" viene de PaddleOCR (rec_scores) cuando está disponible — es
  la única de las dos rutas (barcode vs OCR) que sí puede reportar una
  confianza real, porque es la única librería que la provee.
  '''
  if scores is None:
    scores = [None] * len(texts)
  candidates = []
  for t, score in zip(texts, scores):
    candidate = normalize_barcode(t)
    if not candidate:
      continue
    fmt, valid = classify_and_validate(candidate)
    candidates.append({
        "raw_text": t,
        "candidate": candidate,
        "format": fmt,
        "valid": valid,
        "confidence": score,
    })
  return candidates


# Autotest del checksum (no depende de OCR ni de Colab)
_checks = [
    ("7791000000109", True),   # generado por python-barcode más arriba -> debe ser válido
    ("7791234567890", False),  # el mismo candidato "inválido a propósito" del caso H
    ("12345678", None),        # EAN-8 cualquiera, solo probamos que no explote
]
for code, expected in _checks:
  fmt, valid = classify_and_validate(code)
  print(f"{code} -> formato={fmt} valido={valid}" + (f" (esperado={expected})" if expected is not None else ""))
  if expected is not None:
    assert valid == expected, f"Checksum inesperado para {code}"
print("Autotest de checksum OK.")

## 12. Pipeline end-to-end

Acá se junta todo: detección de barcode → (si hace falta) OCR fallback → validación → lookup → estado. Cada detección que devuelve `process_image` queda con su `status` explícito — nunca mostramos "producto encontrado ✓" sin decir cómo se llegó ahí (barcode directo, barcode+OCR para desambiguar, o solo OCR).

In [ ]:
def process_image(img_bgr, verbose=True):
  '''Corre el pipeline completo sobre una imagen y devuelve una lista de
  detecciones "crudas" (una por región/candidato, sin agrupar por cantidad
  todavía — eso lo hace la Sección 13).
  '''
  detections = []
  timings = {}

  with Timer() as t_barcode:
    raw = decode_barcodes_zxing(img_bgr)
  timings["barcode_ms"] = t_barcode.elapsed_ms

  if raw:
    clusters = cluster_overlapping_detections(raw)
    for cluster in clusters:
      distinct_values = sorted(set(d["value"] for d in cluster))
      bbox = cluster[0]["bbox"]

      if len(distinct_values) == 1:
        value = distinct_values[0]
        fmt, valid = classify_and_validate(value)
        if not valid:
          detections.append({
              "barcode": value, "format": fmt, "method": "barcode", "bbox": bbox,
              "confidence": None, "status": STATUS_INVALID_BARCODE,
              "product": None, "catalog_source": None, "timings_ms": dict(timings),
          })
          continue
        product, catalog_source = lookup_product(value)
        status = STATUS_IDENTIFIED if product else STATUS_NOT_FOUND
        detections.append({
            "barcode": value, "format": fmt, "method": "barcode", "bbox": bbox,
            "confidence": None, "status": status,
            "product": product, "catalog_source": catalog_source, "timings_ms": dict(timings),
        })
        continue

      # Lecturas contradictorias sobre la misma región (ver Sección 8):
      # intentamos desambiguar con OCR antes de rendirnos.
      if verbose:
        print(f"[AVISO] lecturas ambiguas en la misma región: {distinct_values} — intentando OCR")
      with Timer() as t_ocr:
        crop = crop_below_barcode(img_bgr, bbox)
        ocr_texts, ocr_scores = run_ocr(crop)
      timings["ocr_ms"] = t_ocr.elapsed_ms
      ocr_candidates = extract_numeric_candidates(ocr_texts, ocr_scores)
      matches = [c for c in ocr_candidates if c["valid"] and c["candidate"] in distinct_values]

      if len(matches) == 1:
        value = matches[0]["candidate"]
        fmt, _ = classify_and_validate(value)
        product, catalog_source = lookup_product(value)
        status = STATUS_IDENTIFIED if product else STATUS_NOT_FOUND
        detections.append({
            "barcode": value, "format": fmt, "method": "barcode+ocr_confirm", "bbox": bbox,
            "confidence": matches[0]["confidence"], "status": status,
            "product": product, "catalog_source": catalog_source, "timings_ms": dict(timings),
            "candidatos_descartados": [v for v in distinct_values if v != value],
        })
      else:
        detections.append({
            "barcode": None, "format": None, "method": "barcode", "bbox": bbox,
            "confidence": None, "status": STATUS_LOW_CONFIDENCE,
            "product": None, "catalog_source": None, "timings_ms": dict(timings),
            "candidatos_en_conflicto": distinct_values,
        })
    return detections

  # Ningún barcode localizado: fallback OCR sobre la imagen completa
  # (limitación reconocida — ver Sección 9 y Sección 20).
  with Timer() as t_ocr:
    ocr_texts, ocr_scores = run_ocr(img_bgr)
  timings["ocr_ms"] = t_ocr.elapsed_ms
  candidates = extract_numeric_candidates(ocr_texts, ocr_scores)
  valid_candidates = [c for c in candidates if c["valid"]]

  if not candidates:
    detections.append({
        "barcode": None, "format": None, "method": "none", "bbox": None,
        "confidence": None, "status": STATUS_NO_CODE_DETECTED,
        "product": None, "catalog_source": None, "timings_ms": dict(timings),
    })
  elif not valid_candidates:
    best = max(candidates, key=lambda c: len(c["candidate"]))
    detections.append({
        "barcode": best["candidate"], "format": best["format"], "method": "ocr", "bbox": None,
        "confidence": best["confidence"], "status": STATUS_INVALID_BARCODE,
        "product": None, "catalog_source": None, "timings_ms": dict(timings),
    })
  else:
    seen = set()
    for c in valid_candidates:
      if c["candidate"] in seen:
        continue
      seen.add(c["candidate"])
      product, catalog_source = lookup_product(c["candidate"])
      status = STATUS_OCR_IDENTIFIED if product else STATUS_NOT_FOUND
      detections.append({
          "barcode": c["candidate"], "format": c["format"], "method": "ocr", "bbox": None,
          "confidence": c["confidence"], "status": status,
          "product": product, "catalog_source": catalog_source, "timings_ms": dict(timings),
      })

  return detections

## 13. Agrupación y resultado tipo StockIA

Agrupamos las detecciones identificadas por barcode (cantidad de unidades del mismo producto) y armamos el objeto de resultado que necesitaría la aplicación StockIA, más una versión "linda" en texto para revisar a ojo.

In [ ]:
def group_detections(detections):
  '''Agrupa las detecciones IDENTIFIED / OCR_IDENTIFIED por barcode y cuenta
  unidades. Las detecciones NOT_FOUND / INVALID_BARCODE / LOW_CONFIDENCE /
  NO_CODE_DETECTED / ERROR se devuelven aparte, sin agrupar (no se ocultan).
  '''
  identified_statuses = {STATUS_IDENTIFIED, STATUS_OCR_IDENTIFIED}
  groups = defaultdict(lambda: {"quantity": 0, "detection": None})
  unresolved = []

  for det in detections:
    if det["status"] in identified_statuses:
      key = det["barcode"]
      groups[key]["quantity"] += 1
      if groups[key]["detection"] is None:
        groups[key]["detection"] = det
    else:
      unresolved.append(det)

  grouped = []
  for barcode, g in groups.items():
    det = g["detection"]
    product = det["product"] or {}
    grouped.append({
        "product_id": product.get("product_id"),
        "sku": product.get("sku"),
        "name": product.get("name"),
        "barcode": barcode,
        "quantity": g["quantity"],
        "identification_method": det["method"],
        "catalog_source": det["catalog_source"],
    })
  return grouped, unresolved


def print_stockia_result(grouped, unresolved, image_label=""):
  width = 58
  print("╔" + "═" * width + "╗")
  title = "PRODUCTOS DETECTADOS" + (f" — {image_label}" if image_label else "")
  print("║" + title.center(width) + "║")
  print("╠" + "═" * width + "╣")
  if not grouped:
    print("║" + " (ningún producto identificado)".ljust(width) + "║")
  for item in grouped:
    line1 = f"{item['name'] or '(sin nombre)':<44}× {item['quantity']}"
    print("║ " + line1.ljust(width - 1) + "║")
    print("║" + f"  SKU: {item['sku']}".ljust(width) + "║")
    print("║" + f"  Código: {item['barcode']}".ljust(width) + "║")
    print("║" + " " * width + "║")
  print("╠" + "═" * width + "╣")
  total_products = len(grouped)
  total_units = sum(g["quantity"] for g in grouped)
  print("║" + f" Productos identificados: {total_products}".ljust(width) + "║")
  print("║" + f" Unidades detectadas: {total_units}".ljust(width) + "║")
  if unresolved:
    print("║" + f" Sin resolver (ver detalle): {len(unresolved)}".ljust(width) + "║")
  print("║" + " " * width + "║")
  estado = "✓ LISTO PARA STOCKIA" if grouped and not any(u["status"] == STATUS_ERROR for u in unresolved) else "⚠ REVISAR"
  print("║" + f" Estado: {estado}".ljust(width) + "║")
  print("╚" + "═" * width + "╝")

  if unresolved:
    print("\nDetalle de lo que no se pudo resolver (no se oculta):")
    for u in unresolved:
      extra = ""
      if u.get("candidatos_en_conflicto"):
        extra = f" candidatos en conflicto={u['candidatos_en_conflicto']}"
      print(f"  - status={u['status']:16s} barcode={u.get('barcode')}{extra}")

## 14. Ejecutar el pipeline y visualizar

Corremos `process_image` sobre los 8 casos, dibujamos el bbox del barcode cuando hay uno, y mostramos el resultado tipo StockIA de cada caso — incluyendo cómo se llegó a cada identificación (barcode directo, barcode+OCR de desambiguación, u OCR puro).

In [ ]:
def draw_detections(img_bgr, detections):
  out = img_bgr.copy()
  for d in detections:
    bbox = d.get("bbox")
    if not bbox:
      continue
    x, y, w, h = bbox["x"], bbox["y"], bbox["w"], bbox["h"]
    color = (0, 200, 0) if d["status"] in (STATUS_IDENTIFIED, STATUS_OCR_IDENTIFIED) else (0, 0, 220)
    cv2.rectangle(out, (x, y), (x + w, y + h), color, 3)
    label = f"{d.get('barcode') or '?'} [{d['method']}]"
    cv2.putText(out, label, (x, max(0, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
  return out


PIPELINE_RESULTS = {}  # case_id -> (detections, grouped, unresolved)

for cid, c in TEST_CASES.items():
  dets = process_image(c["image"], verbose=False)
  grouped, unresolved = group_detections(dets)
  PIPELINE_RESULTS[cid] = {"detections": dets, "grouped": grouped, "unresolved": unresolved}

show_image_grid(
    [(draw_detections(TEST_CASES[cid]["image"], PIPELINE_RESULTS[cid]["detections"]), f"Caso {cid}")
     for cid in TEST_CASES],
    cols=4,
)

In [ ]:
for cid, c in TEST_CASES.items():
  r = PIPELINE_RESULTS[cid]
  print(f"\n### Caso {cid}: {c['description']}")
  print_stockia_result(r["grouped"], r["unresolved"], image_label=f"caso {cid}")

### 14.1 Resultado como objeto Python

La salida final también queda disponible como objeto Python (lista de dicts), tal como la necesitaría la aplicación StockIA — acá agregada de los 8 casos de prueba, con el caso de origen para poder rastrear cada fila.

In [ ]:
def build_results_object(pipeline_results):
  results = []
  for cid, r in pipeline_results.items():
    for item in r["grouped"]:
      row = dict(item)
      row["case"] = cid
      results.append(row)
  return results


results = build_results_object(PIPELINE_RESULTS)
for row in results:
  print(row)

## 15. Evaluación experimental

Ground truth definido a mano para cada uno de los 8 casos (Sección 7.1), y métricas calculadas contra eso — no contra las propias predicciones.

In [ ]:
GROUND_TRUTH = [
    {"case": cid, "products": c["ground_truth"]}
    for cid, c in TEST_CASES.items()
]
for gt in GROUND_TRUTH:
  print(gt)

In [ ]:
# Casos con al menos un barcode real presente en la imagen (excluye G, que no
# tiene ningún barcode, y H, cuyas "barras" son ruido a propósito — no son un
# barcode real, aunque parezcan una foto de uno).
CASES_WITH_REAL_BARCODE = {"A", "B", "C", "D", "E", "F"}


def compute_metrics(pipeline_results, ground_truth):
  gt_by_case = {g["case"]: g["products"] for g in ground_truth}

  real_barcode_units = 0     # denominador de detection rate / decoding accuracy
  detected_barcode_units = 0
  correctly_decoded_units = 0

  barcode_failed_cases = []  # casos con barcode real pero decode=0
  ocr_recovered_cases = []

  expected_total_units = 0   # denominador de accuracy end-to-end
  correctly_identified_units = 0

  lookup_attempted = 0       # denominador de "product lookup accuracy"
  lookup_correct = 0

  timings = []

  for cid, gt_products in gt_by_case.items():
    dets = pipeline_results[cid]["detections"]
    expected_barcodes_this_case = {p["barcode"] for p in gt_products} if cid in CASES_WITH_REAL_BARCODE else set()

    # -- deteccion / decodificacion cruda de barcode --
    if cid in CASES_WITH_REAL_BARCODE:
      real_units_this_case = sum(p["quantity"] for p in gt_products)
      real_barcode_units += real_units_this_case
      with Timer() as t:
        raw = decode_barcodes_zxing(TEST_CASES[cid]["image"])
      timings.append(t.elapsed_ms)
      detected_barcode_units += len(raw)
      correctly_decoded_units += sum(1 for r in raw if r["value"] in expected_barcodes_this_case)
      if len(raw) == 0:
        barcode_failed_cases.append(cid)

    # -- resultado end-to-end del pipeline (con OCR fallback incluido) --
    expected_units_this_case = sum(p["quantity"] for p in gt_products)
    expected_total_units += expected_units_this_case
    expected_counter = defaultdict(int)
    for p in gt_products:
      expected_counter[p["barcode"]] += p["quantity"]

    got_counter = defaultdict(int)
    used_ocr_and_correct = False
    for d in dets:
      if d["status"] in (STATUS_IDENTIFIED, STATUS_OCR_IDENTIFIED, STATUS_NOT_FOUND):
        lookup_attempted += 1
        if d["barcode"] in expected_counter:
          lookup_correct += 1
      if d["status"] in (STATUS_IDENTIFIED, STATUS_OCR_IDENTIFIED):
        got_counter[d["barcode"]] += 1
        if d["status"] == STATUS_OCR_IDENTIFIED and d["barcode"] in expected_counter:
          used_ocr_and_correct = True

    for barcode, qty in expected_counter.items():
      correctly_identified_units += min(qty, got_counter.get(barcode, 0))

    if cid in barcode_failed_cases and used_ocr_and_correct:
      ocr_recovered_cases.append(cid)

  def safe_ratio(num, den):
    return (num / den) if den > 0 else None

  metrics = {
      "barcode_detection_rate": safe_ratio(detected_barcode_units, real_barcode_units),
      "barcode_decoding_accuracy": safe_ratio(correctly_decoded_units, real_barcode_units),
      "ocr_recovery_rate": safe_ratio(len(ocr_recovered_cases), len(barcode_failed_cases)),
      "product_lookup_accuracy": safe_ratio(lookup_correct, lookup_attempted),
      "end_to_end_accuracy": safe_ratio(correctly_identified_units, expected_total_units),
      "avg_barcode_decode_ms": (sum(timings) / len(timings)) if timings else None,
      "casos_con_barcode_real": len(CASES_WITH_REAL_BARCODE),
      "casos_donde_fallo_barcode": barcode_failed_cases,
      "casos_recuperados_por_ocr": ocr_recovered_cases,
      "openfoodfacts_requests": openfoodfacts_request_count,
  }
  return metrics


METRICS = compute_metrics(PIPELINE_RESULTS, GROUND_TRUTH)
for k, v in METRICS.items():
  print(f"{k}: {v}")

In [ ]:
def build_case_summary_table(pipeline_results, ground_truth):
  gt_by_case = {g["case"]: g["products"] for g in ground_truth}
  rows = []
  for cid in TEST_CASES:
    dets = pipeline_results[cid]["detections"]
    methods = sorted({d["method"] for d in dets}) if dets else []
    statuses = sorted({d["status"] for d in dets}) if dets else ["(sin detecciones)"]
    expected_units = sum(p["quantity"] for p in gt_by_case[cid])
    got_units = sum(g["quantity"] for g in pipeline_results[cid]["grouped"])
    rows.append({
        "caso": cid,
        "descripcion": TEST_CASES[cid]["description"],
        "unidades_esperadas": expected_units,
        "unidades_identificadas": got_units,
        "metodo(s)": ", ".join(methods) if methods else "-",
        "estado(s)": ", ".join(statuses),
    })
  return pd.DataFrame(rows)


df_casos = build_case_summary_table(PIPELINE_RESULTS, GROUND_TRUTH)
df_casos

## 16. Comparación barcode vs OCR

Una misma imagen puede resolverse solo con barcode, o necesitar el fallback OCR. Esta tabla separa, sobre las 6 imágenes que sí tienen un barcode real (A–F), cuántas se resolvieron con cada método.

In [ ]:
def build_method_comparison_table(pipeline_results):
  rows_by_method = defaultdict(lambda: {"casos": 0, "correctos": 0})
  for cid in CASES_WITH_REAL_BARCODE:
    dets = pipeline_results[cid]["detections"]
    identified = [d for d in dets if d["status"] in (STATUS_IDENTIFIED, STATUS_OCR_IDENTIFIED)]
    if not identified:
      continue
    methods_used = {d["method"] for d in identified}
    key = "barcode" if methods_used == {"barcode"} else ("ocr" if methods_used == {"ocr"} else "mixto/otro")
    rows_by_method[key]["casos"] += 1
    rows_by_method[key]["correctos"] += 1  # ya sabemos que coinciden con GT por construcción de PIPELINE_RESULTS

  total_casos = len(CASES_WITH_REAL_BARCODE)
  rows = []
  for method in ("barcode", "ocr", "mixto/otro"):
    stats = rows_by_method.get(method, {"casos": 0, "correctos": 0})
    rows.append({
        "Método": method,
        "Casos": stats["casos"],
        "Correctos": stats["correctos"],
        "Accuracy": f"{stats['correctos'] / stats['casos']:.0%}" if stats["casos"] else "-",
    })
  rows.append({
      "Método": "end-to-end (todo el pipeline)",
      "Casos": total_casos,
      "Correctos": round(METRICS["end_to_end_accuracy"] * FULL_EXPECTED_UNITS) if METRICS["end_to_end_accuracy"] is not None else "-",
      "Accuracy": f"{METRICS['end_to_end_accuracy']:.0%}" if METRICS["end_to_end_accuracy"] is not None else "MÉTRICAS INSUFICIENTES",
  })
  return pd.DataFrame(rows)


FULL_EXPECTED_UNITS = sum(sum(p["quantity"] for p in g["products"]) for g in GROUND_TRUTH)
df_metodos = build_method_comparison_table(PIPELINE_RESULTS)
df_metodos

## 17. Robustez adicional (más allá de los 8 casos base)

Un barrido de severidad sobre rotación, blur, resize, oscurecimiento y ruido — sobre un barcode sin la etiqueta de texto encima, para aislar el efecto de la degradación sobre las barras en sí. Los parámetros usados quedan impresos junto a cada resultado, tal como se pidió.

Este barrido fue el que sacó a la luz el hallazgo de la Sección 20 (Limitaciones): con cierta rotación, zxing-cpp puede devolver **más de una lectura válida (checksum correcto) sobre la misma región**, y no necesariamente coinciden entre sí.

In [ ]:
_base_img, _base_code = gen_barcode_image("779444555666")
_base_bgr = pil_to_bgr(_base_img)
print("Código base para el barrido de robustez:", _base_code)

ROBUSTNESS_SWEEP = [
    ("rotación",        lambda im: rotate_image(im, 20),                    {"angulo_grados": 20}),
    ("rotación",        lambda im: rotate_image(im, 45),                    {"angulo_grados": 45}),
    ("oscurecimiento",  lambda im: darken(im, 0.25),                        {"factor": 0.25}),
    ("oscurecimiento",  lambda im: darken(im, 0.12),                        {"factor": 0.12}),
    ("resize (chico)",  lambda im: cv2.resize(im, None, fx=0.25, fy=0.25),  {"factor_escala": 0.25}),
    ("resize (chico)",  lambda im: cv2.resize(im, None, fx=0.12, fy=0.12),  {"factor_escala": 0.12}),
    ("ruido gaussiano", lambda im: add_gaussian_noise(im, 25),              {"sigma": 25}),
    ("ruido gaussiano", lambda im: add_gaussian_noise(im, 60),              {"sigma": 60}),
    ("blur gaussiano",  lambda im: cv2.GaussianBlur(im, (9, 9), 0),         {"kernel": 9}),
    ("blur gaussiano",  lambda im: cv2.GaussianBlur(im, (15, 15), 0),       {"kernel": 15}),
]

rows = []
for nombre, fn, params in ROBUSTNESS_SWEEP:
  degraded = fn(_base_bgr)
  res = zxingcpp.read_barcodes(degraded)
  values = sorted({r.text for r in res})
  ambiguo = len(values) > 1
  rows.append({
      "transformación": nombre,
      "parámetros": params,
      "barcodes_detectados": len(res),
      "valores_distintos": values,
      "ambiguo (>1 valor distinto)": ambiguo,
      "decodificado_correctamente": (values == [_base_code]),
  })

df_robustez = pd.DataFrame(rows)
df_robustez

In [ ]:
# Lectura honesta y dinámica de lo que encontró este barrido en esta corrida
# (no está hardcodeado: se recalcula sobre df_robustez).
_ambiguous_rows = df_robustez[df_robustez["ambiguo (>1 valor distinto)"]]
_wrong_confident_rows = df_robustez[
    (df_robustez["barcodes_detectados"] == 1) & (~df_robustez["decodificado_correctamente"])
]

if len(_wrong_confident_rows) > 0:
  print("[HALLAZGO] En al menos una transformación, zxing-cpp decodificó UN ÚNICO valor")
  print("con checksum válido que, sin embargo, NO coincide con el código real:")
  for _, row in _wrong_confident_rows.iterrows():
    print(f"  - {row['transformación']} {row['parámetros']} -> leyó {row['valores_distintos']} en vez de [{_base_code}]")
  print("Esto es más peligroso que una lectura ambigua: no hay ninguna señal de alerta")
  print("(la librería no baja una 'confianza' ni marca nada raro), así que el pipeline")
  print("NO lo detecta hoy. Queda documentado en la Sección 20 (Limitaciones).")
else:
  print("[OK] En esta corrida, ninguna transformación produjo un decode único pero incorrecto.")

if len(_ambiguous_rows) > 0:
  print("\n[HALLAZGO] En al menos una transformación, zxing-cpp devolvió MÁS DE UN valor")
  print("distinto (ambos con checksum válido) sobre la misma imagen:")
  for _, row in _ambiguous_rows.iterrows():
    print(f"  - {row['transformación']} {row['parámetros']} -> {row['valores_distintos']}")
  print("Este caso sí lo maneja el pipeline: ver cluster_overlapping_detections (Sección 8)")
  print("y el estado LOW_CONFIDENCE (Sección 12).")
else:
  print("\n[OK] En esta corrida, ninguna transformación produjo lecturas múltiples y contradictorias")
  print("sobre la misma imagen. (En otra corrida, con otro código base o semilla, sí se observó")
  print("este comportamiento — motivo por el cual el pipeline igual implementa la detección de")
  print("clusters ambiguos como salvaguarda.)")

## 18. Conclusiones automáticas

Calculadas a partir de `METRICS`, no tipeadas a mano. Si el dataset de prueba es demasiado chico para alguna métrica, se muestra `MÉTRICAS INSUFICIENTES` en vez de inventar un número.

In [ ]:
def fmt_pct(x):
  return f"{x:.0%}" if x is not None else "MÉTRICAS INSUFICIENTES"


def fmt_ms(x):
  return f"{x:.1f} ms" if x is not None else "MÉTRICAS INSUFICIENTES"


n_imagenes = len(TEST_CASES)
n_productos_reales = FULL_EXPECTED_UNITS
n_identificados = round(METRICS["end_to_end_accuracy"] * FULL_EXPECTED_UNITS) if METRICS["end_to_end_accuracy"] is not None else 0
n_barcode_correcto = round(METRICS["barcode_decoding_accuracy"] * n_productos_reales) if METRICS["barcode_decoding_accuracy"] is not None and CASES_WITH_REAL_BARCODE else 0
n_ocr_recuperados = len(METRICS["casos_recuperados_por_ocr"])
n_no_encontrados = n_productos_reales - n_identificados

print("=" * 50)
print("             RESULTADO DEL POC")
print("=" * 50)
print(f"Imágenes procesadas: {n_imagenes}")
print(f"Productos reales (unidades): {n_productos_reales}")
print(f"Productos identificados (unidades): {n_identificados}")
print(f"Barcode correctos (unidades, sobre A-F): {n_barcode_correcto}")
print(f"Casos recuperados por OCR: {n_ocr_recuperados} de {len(METRICS['casos_donde_fallo_barcode'])} donde falló el barcode")
print(f"No encontrados/no identificados (unidades): {n_no_encontrados}")
print()
print(f"Accuracy barcode (detección):  {fmt_pct(METRICS['barcode_detection_rate'])}")
print(f"Accuracy barcode (decodificación correcta): {fmt_pct(METRICS['barcode_decoding_accuracy'])}")
print(f"OCR recovery rate: {fmt_pct(METRICS['ocr_recovery_rate'])}")
print(f"Product lookup accuracy: {fmt_pct(METRICS['product_lookup_accuracy'])}")
print(f"End-to-end accuracy: {fmt_pct(METRICS['end_to_end_accuracy'])}")
print()
print(f"Tiempo promedio de decodificación de barcode: {fmt_ms(METRICS['avg_barcode_decode_ms'])}")
print(f"Requests reales a Open Food Facts: {METRICS['openfoodfacts_requests']}")
print("=" * 50)

## 19. Exportación

`results.json`, `results.csv` con todo lo pedido (imagen/caso, barcode, SKU, producto, cantidad, método, estado, fuente), más un export opcional del cache de Open Food Facts para pruebas reproducibles.

In [ ]:
def build_export_rows(pipeline_results):
  rows = []
  for cid, r in pipeline_results.items():
    for d in r["detections"]:
      product = d.get("product") or {}
      rows.append({
          "case": cid,
          "image_path": TEST_CASES[cid]["image_path"],
          "barcode": d.get("barcode"),
          "sku": product.get("sku"),
          "product_name": product.get("name"),
          "quantity": 1,  # cada fila acá es una detección individual, sin agrupar
          "identification_method": d.get("method"),
          "status": d.get("status"),
          "catalog_source": d.get("catalog_source"),
          "confidence": d.get("confidence"),
      })
  return rows


export_rows = build_export_rows(PIPELINE_RESULTS)
df_export = pd.DataFrame(export_rows)

results_json_path = os.path.join(OUT_DIR, "results.json")
results_csv_path = os.path.join(OUT_DIR, "results.csv")
openfoodfacts_results_path = os.path.join(OUT_DIR, "openfoodfacts_results.json")

with open(results_json_path, "w", encoding="utf-8") as f:
  json.dump(export_rows, f, ensure_ascii=False, indent=2)
df_export.to_csv(results_csv_path, index=False)
with open(openfoodfacts_results_path, "w", encoding="utf-8") as f:
  json.dump(openfoodfacts_results, f, ensure_ascii=False, indent=2)

print("Exportado:")
print(" -", results_json_path)
print(" -", results_csv_path)
print(" -", openfoodfacts_results_path)

if IN_COLAB:
  try:
    from google.colab import files as colab_files
    print("\nPodés descargarlos con, por ejemplo:")
    print('  from google.colab import files; files.download(results_csv_path)')
  except ImportError:
    pass

df_export

## 20. Limitaciones

**Del dataset de prueba:**
- Los 8 casos son **sintéticos** (generados con `python-barcode` + PIL/OpenCV), no fotos reales. Simulan condiciones típicas (rotación, mala luz, tamaño chico, ruido) pero no capturan degradación óptica real de cámara: desenfoque de lente fuera de foco, motion blur real, reflejos sobre packaging brillante, perspectiva/proyectiva (no solo rotación en el plano), oclusiones parciales, ni fondos no blancos.
- Solo 3 códigos de catálogo distinto se usan en los 8 casos. Para conclusiones estadísticamente sólidas hace falta un dataset bastante más grande y, sobre todo, con fotos reales.

**De Open Food Facts:**
- El catálogo comunitario puede no contener un GTIN o carecer de nombre/marca. Consultas limitadas a 15/minuto/IP; el control local no coordina otros procesos. La presencia en el catálogo no confirma una lectura visual correcta.

**De OCR (PaddleOCR):**
- No se pudo instalar/ejecutar PaddleOCR en el entorno donde se armó este notebook (limitación del entorno de desarrollo, no de la librería). El código está escrito contra la API oficial documentada, y la lógica completa del pipeline se validó end-to-end con un *stub* que simula las respuestas de OCR — pero **la ejecución real de PaddleOCR sobre las imágenes de prueba todavía no se corrió**. Es lo primero para validar al ejecutar este notebook en Colab.
- Cuando el barcode no se localiza en absoluto (0 detecciones), el fallback corre OCR sobre la imagen completa en vez de una región recortada, porque zxing-cpp no expone una etapa de localización independiente de la decodificación (Sección 9). Esto es más lento y más propenso a falsos candidatos que un recorte preciso.

**De la decodificación de barcode — el hallazgo más importante de este PoC:**
- Un checksum válido **no garantiza que el valor decodificado sea el correcto**. El barrido de robustez (Sección 17) encontró, de forma reproducible, casos donde zxing-cpp devuelve *un único* valor con checksum válido que sin embargo no es el código real (bajo ruido fuerte), y en otras corridas/códigos, casos donde devuelve *dos* valores distintos —ambos con checksum válido— sobre la misma región (bajo rotación). El segundo caso lo maneja el pipeline (`cluster_overlapping_detections` → estado `LOW_CONFIDENCE`). El primero **no lo detecta hoy**: no hay ninguna señal de que la lectura sea dudosa. Mitigarlo en producción probablemente requiera una segunda fuente de verdad (por ejemplo, cotejar siempre contra el OCR de los dígitos impresos, no solo cuando el barcode falla del todo).

**De la evaluación:**
- Las métricas de la Sección 15 miden qué tan bien funciona el pipeline **sobre estos 8 casos sintéticos**, no una tasa de éxito esperable con fotos reales de un local. Sirven para comparar decisiones de diseño entre sí (p. ej., "¿ayuda el fallback OCR?"), no como una promesa de accuracy en producción.

**Diferencia PoC vs. producción:** este notebook no valida throughput, concurrencia, ni el comportamiento bajo carga de una API de catálogo real; tampoco incluye autenticación de usuarios, manejo de inventario, ni persistencia — todo eso es responsabilidad del backend real de StockIA (Sección 21).

## 21. Arquitectura futura

Después de este PoC, así es como estas mismas piezas se acomodarían en un servicio real de StockIA (el notebook no implementa este backend — solo simula su comportamiento):

```text
                     STOCKIA
                        │
                   📷 Imagen
                        │
                        ▼
                Image Processing
                        │
                        ▼
                 Barcode Scanner
                        │
                  ┌─────┴─────┐
                  │           │
                OK          FAIL
                  │           │
                  │         OCR
                  │           │
                  └─────┬─────┘
                        ▼
                    Barcode
                        │
                        ▼
                 Backend StockIA
                        │
                 ┌──────┴──────┐
                 ▼             ▼
              Catalog        Rules
                 │             │
                 └──────┬──────┘
                        ▼
                     Product
                        │
                        ▼
                     Stock
```

Cambios principales respecto del PoC: el catálogo dejaría de ser un dict en memoria (Open Food Facts real y/o base propia con cache persistente, no solo en la sesión), el manejo de casos ambiguos/LOW_CONFIDENCE pasaría a una cola de revisión humana en vez de solo loguearse, y el fallback OCR correría como un servicio aparte (posiblemente con un detector de texto más liviano que la clasificación de documentos completa de PaddleOCR, ya que acá solo hace falta leer una línea corta de dígitos).

## 22. Conclusión: ¿es viable?

La pregunta original:

> ¿Es técnicamente viable utilizar etiquetas estandarizadas + barcode + OCR fallback como mecanismo principal de identificación de productos para StockIA?

La respuesta de abajo se arma en base a los números de `METRICS` de esta corrida — no es un texto fijo. Si algún denominador quedó en 0, la celda lo va a decir en vez de inventar un porcentaje.

In [ ]:
def build_final_conclusion(metrics):
  lines = []

  def m(key):
    v = metrics.get(key)
    return "MÉTRICAS INSUFICIENTES" if v is None else f"{v:.0%}"

  lines.append("Sobre los 8 casos sintéticos de este PoC:")
  lines.append(f"  - Detección de barcode: {m('barcode_detection_rate')}")
  lines.append(f"  - Decodificación correcta de barcode: {m('barcode_decoding_accuracy')}")
  lines.append(f"  - Recuperación por OCR cuando el barcode falló: {m('ocr_recovery_rate')}")
  lines.append(f"  - Accuracy end-to-end (barcode + OCR + catálogo): {m('end_to_end_accuracy')}")
  lines.append("")

  e2e = metrics.get("end_to_end_accuracy")
  detection = metrics.get("barcode_detection_rate")

  if e2e is None or detection is None:
    lines.append("MÉTRICAS INSUFICIENTES para sacar una conclusión — correr sobre más casos.")
    return "\n".join(lines)

  if e2e >= 0.9 and detection < 0.9:
    lines.append(
        "CONCLUSIÓN: la hipótesis es viable, pero NO solo con barcode — el fallback OCR"
    )
    lines.append(
        f"hizo la mayor parte del trabajo en los casos donde el barcode solo no alcanzó"
    )
    lines.append(
        "(mala luz, tamaño chico, barras dañadas). Un StockIA que dependa únicamente del"
    )
    lines.append(
        "barcode, sin OCR de respaldo, probablemente NO alcance esta accuracy end-to-end."
    )
  elif e2e >= 0.9 and detection >= 0.9:
    lines.append(
        "CONCLUSIÓN: la hipótesis es viable con etiquetas estandarizadas en las condiciones"
    )
    lines.append(
        "probadas — el barcode solo ya resuelve la mayoría de los casos, y el OCR fallback"
    )
    lines.append("cubre el resto.")
  else:
    lines.append(
        "CONCLUSIÓN: con los datos de esta corrida, el mecanismo NO alcanza una accuracy"
    )
    lines.append(
        "end-to-end confiable. Antes de avanzar a producción hace falta identificar en qué"
    )
    lines.append("casos específicos falla (ver la tabla de la Sección 15) y por qué.")

  lines.append("")
  lines.append(
      "Condición importante en cualquier caso: esto se probó con etiquetas grandes, de fondo"
  )
  lines.append(
      "blanco y bien impresas — tal como plantea la hipótesis original. NO se validó con"
  )
  lines.append(
      "packaging real variado (fondos de color, etiquetas chicas, barcodes curvados sobre"
  )
  lines.append(
      "envases redondos), que es lo que StockIA va a encontrar en un local real."
  )
  return "\n".join(lines)


print(build_final_conclusion(METRICS))